# Entregável 9 — Seleção de Atributos

**Disciplina:** Aquisição e Processamento de Biossinais  
**Equipe:** José Ferreira Lessa & Matheus Rocha Gomes da Silva  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL — A Large Publicly Available Electrocardiography Dataset (PhysioNet)  
**Referência:** Wagner et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.  
**Data:** Abril e Maio de 2026

---

## Objetivo

Este notebook realiza a **Seleção de Atributos** (*feature selection*) sobre o dataset de features engenhadas produzido no Entregável 7 (`features_engineered.parquet`), com o propósito de identificar e reter o subconjunto de features com maior poder discriminativo para o problema de classificação de ECG.

A seleção de atributos é uma etapa distinta — e complementar — à redução de dimensionalidade realizada no Entregável 8. Enquanto o PCA projeta o espaço original em novas direções abstratas (componentes), a seleção de atributos mantém as variáveis originais, preservando interpretabilidade clínica. Cada feature selecionada continua sendo, por exemplo, "potência espectral do QRS na derivação V5" — e não uma combinação linear de centenas de outras.

Além disso, a seleção reduz o risco de **overfitting**, remove features redundantes que aumentam custo computacional sem agregar informação nova, e melhora a estabilidade dos modelos de classificação ao eliminar dimensões com alta variância, mas baixa relevância discriminativa.

Três famílias de métodos são exploradas neste entregável:

1. **Filter Methods:** avaliam features individualmente com base em critérios estatísticos — rápidos e independentes do classificador. Aplicamos **ANOVA F-score**, **Informação Mútua** e **ReliefF**.
2. **Wrapper Methods:** avaliam subconjuntos de features treinando um estimador — captura interações entre features. Aplicamos **Sequential Forward Selection (SFS)** e **Sequential Backward Elimination (SBE)**.
3. **Embedded Methods:** a seleção ocorre durante o treinamento do próprio modelo. Aplicamos **LASSO (Regularização L1)** e **Importância por Random Forest**.

Por fim, a validação estatística formal de cada feature é realizada por meio de testes de hipótese, correção para múltiplas comparações e cálculo de effect size — garantindo que a seleção final tenha suporte estatístico rigoroso.

O entregável é organizado em:

1. **Importações, Configurações e Dependências**
2. **Carregamento e Inspeção do Dataset**
3. **Filter Methods**
   - 3.1 ANOVA F-score
   - 3.2 Informação Mútua
   - 3.3 ReliefF
4. **Wrapper Methods**
   - 4.1 Sequential Forward Selection (SFS)
   - 4.2 Sequential Backward Elimination (SBE)
5. **Embedded Methods**
   - 5.1 LASSO (Regularização L1)
   - 5.2 Importância por Random Forest
6. **Validação Estatística por Feature**
   - 6.1 Testes de Hipótese por Feature
   - 6.2 Correção de Múltiplas Comparações
   - 6.3 Effect Size (Cohen's d / Eta-quadrado)
7. **Consolidação e Ranking Final dos Atributos**
8. **Dataset Final Selecionado e Persistência**


---

## 1. Importações, Configurações e Dependências

Bibliotecas específicas deste notebook:

- **`sklearn.feature_selection`:** implementações de filter (f_classif, mutual_info_classif), wrapper (SequentialFeatureSelector) e embedded (SelectFromModel).
- **`sklearn.linear_model.LassoCV`:** LASSO com cross-validation automática do hiperparâmetro λ, evitando ajuste manual de regularização.
- **`sklearn.ensemble.RandomForestClassifier`:** usado como estimador para extração de importância via impureza de Gini — robusto e não-paramétrico.
- **`scipy.stats`:** testes de hipótese por feature (Kruskal-Wallis e Mann-Whitney com todas as comparações par-a-par).
- **`statsmodels.stats.multitest`:** correção de Bonferroni e FDR (Benjamini-Hochberg) para o problema de múltiplas comparações.
- **`skrebate`:** implementação do algoritmo ReliefF para seleção baseada em relevância relacional entre instâncias.


In [ ]:
import os
import ast
import gc
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations

# Estatística
from scipy.stats import (kruskal, mannwhitneyu, f_oneway,
                         shapiro, levene)
from statsmodels.stats.multitest import multipletests

# Scikit-learn — seleção e modelos
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import (
    f_classif, mutual_info_classif,
    SequentialFeatureSelector, SelectFromModel,
    RFE
)
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

# ReliefF
try:
    from skrebate import ReliefF
    RELIEFF_OK = True
except ImportError:
    RELIEFF_OK = False
    print("Aviso: skrebate nao instalado. ReliefF sera ignorado.")
    print("Instale com: pip install skrebate")

from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

np.random.seed(42)


---

## 2. Carregamento e Inspeção do Dataset


In [ ]:
FOLDS_TREINO = [1, 2, 3, 4, 5, 6, 7, 8]
FOLD_VAL     = 9
FOLD_TEST    = 10

DIR_IN_D7 = Path('../../entregavel-7/outputs/')
FIGS_DIR  = Path('../figuras/')
OUT_DIR   = Path('../outputs/')

for d in [FIGS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

META_COLS = ['ecg_id', 'patient_id', 'strat_fold', 'quality_class',
             'superclasses_clean', 'primary_class', 'n_superclasses', 'split']

parquet_path = DIR_IN_D7 / 'features_engineered.parquet'
if not parquet_path.exists():
    raise FileNotFoundError(
        f'Arquivo nao encontrado: {parquet_path}\n'
        'Execute o Entregavel 7 antes de prosseguir.'
    )

print('Carregando features engenhadas do Entregavel 7...')
df = pd.read_parquet(str(parquet_path))

if 'superclasses_clean' in df.columns and isinstance(df['superclasses_clean'].iloc[0], str):
    df['superclasses_clean'] = df['superclasses_clean'].apply(ast.literal_eval)

if 'primary_class' not in df.columns:
    df['primary_class'] = df['superclasses_clean'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'UNKNOWN'
    )

META_COLS    = [c for c in META_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in META_COLS]
mask_treino  = df['strat_fold'].isin(FOLDS_TREINO)

print(f'Dataset carregado   : {df.shape}')
print(f'Features            : {len(feature_cols)}')
print(f'Registros de treino : {mask_treino.sum()}')
print(f'Registros totais    : {len(df)}')


In [ ]:
# Inspeção básica do dataset e das classes

classes_ok = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
df_treino  = df[mask_treino].copy()

print('Distribuição de classes no conjunto de treino:')
dist = (df_treino['primary_class']
        .value_counts()
        .rename_axis('Classe')
        .reset_index(name='N')
        .assign(Percentual=lambda x: (x['N'] / x['N'].sum() * 100).round(2))
       )
display(dist)

print()
print(f'Features nulas (treino): {df_treino[feature_cols].isnull().sum().sum()}')
print(f'Features constantes    : {(df_treino[feature_cols].std() == 0).sum()}')


In [ ]:
# Prepara X e y para o conjunto de treino — usados em todas as etapas seguintes

X_treino_raw = df_treino[feature_cols].values
y_treino_raw = df_treino['primary_class'].values

# Codificação numérica das classes (necessária para LASSO e RF)
le = LabelEncoder()
y_treino_enc = le.fit_transform(y_treino_raw)

# Padronização (fit apenas no treino — mesma filosofia do E8)
scaler = StandardScaler()
X_treino = scaler.fit_transform(X_treino_raw)

# Aplica ao dataset inteiro para persistência posterior
X_todos = scaler.transform(df[feature_cols].values)

print(f'X_treino : {X_treino.shape}')
print(f'y_treino : {y_treino_enc.shape}  | classes: {le.classes_}')


---

## 3. Filter Methods

### Fundamentação

Os filter methods avaliam a relevância de cada feature **independentemente do classificador**, com base em critérios estatísticos calculados sobre a distribuição da feature em relação à variável resposta. Por serem desacoplados do modelo, são computacionalmente eficientes e podem ser aplicados a datasets de alta dimensão.

A principal limitação dessa abordagem é justamente essa independência: features individualmente fracas podem ser coletivamente informativas devido a interações, e isso não é capturado pelos filtros. Por isso, os resultados aqui são usados principalmente para um **ranking exploratório**, que servirá de comparação e complemento às abordagens wrapper e embedded.

Três técnicas são empregadas nesta seção:

- **ANOVA F-score:** quantifica a separação entre médias das classes em relação à variância intragrupo. Assume que as distribuições por classe são aproximadamente normais e que as variâncias são homogêneas (condições avaliadas nos entregáveis anteriores).
- **Informação Mútua (MI):** captura dependências não-lineares entre a feature e o rótulo. Não faz suposições paramétrics sobre a distribuição dos dados — mais robusto quando a relação feature–classe é complexa.
- **ReliefF:** avalia a relevância de cada feature com base na proximidade entre instâncias de mesma classe (*near-hit*) e de classes diferentes (*near-miss*). Captura interações locais entre features e é especialmente útil em problemas com múltiplas classes.


### 3.1 ANOVA F-score

A **ANOVA unidirecional** testa se as médias de uma feature diferem significativamente entre as $C$ classes. Sob a hipótese nula, a distribuição dos dados é igual para todas as classes. O F-statistic é definido como a razão entre a variância **intergrupos** (explicada pela classe) e a variância **intragrupo** (residual):

$$F = \frac{\text{Variância entre grupos} / (C - 1)}{\text{Variância dentro dos grupos} / (N - C)}$$

Um F alto indica que a feature separa bem as classes em termos de média. O p-value associado indica a probabilidade de observar esse F sob $H_0$ — features com p-value baixo (após correção para múltiplas comparações) são candidatas fortes à seleção.

**Atenção:** o F-score é sensível apenas a diferenças de média. Uma feature pode ter distribuições muito diferentes entre classes sem mudança de média (ex: classes com mesma média mas variâncias diferentes), e isso não seria capturado pelo ANOVA.


In [ ]:
# ANOVA F-score sobre o conjunto de treino padronizado

F_vals, p_vals_anova = f_classif(X_treino, y_treino_raw)

df_anova = pd.DataFrame({
    'feature'  : feature_cols,
    'F_score'  : F_vals,
    'p_value'  : p_vals_anova,
}).sort_values('F_score', ascending=False).reset_index(drop=True)

df_anova['rank_anova'] = df_anova.index + 1

print(f'Top 20 features por ANOVA F-score:')
display(df_anova.head(20).round(4))
print()
print(f'Features com p-value < 0.05: {(df_anova["p_value"] < 0.05).sum()} de {len(feature_cols)}')
print(f'Features com p-value < 0.01: {(df_anova["p_value"] < 0.01).sum()} de {len(feature_cols)}')


In [ ]:
# Barplot — Top 30 por F-score

top30_anova = df_anova.head(30)

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(top30_anova['feature'][::-1], top30_anova['F_score'][::-1],
               color=plt.cm.RdYlGn(np.linspace(0.3, 0.9, 30)))
ax.set_xlabel('ANOVA F-score', fontsize=11)
ax.set_title('Top 30 Features — ANOVA F-score (conjunto de treino)', fontsize=12)
ax.axvline(top30_anova['F_score'].mean(), color='navy', ls='--', lw=1.2,
           label=f'Média top30: {top30_anova["F_score"].mean():.1f}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_anova_top30.png', dpi=150, bbox_inches='tight')
plt.show()


**Comentários sobre a subseção 3.1 — ANOVA F-score:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quais domínios (temporal, espectral, morfológico, não-linear) dominam o topo do ranking? Há concentração em um único domínio ou a lista é diversificada?
> - O F-score decai rapidamente (cauda curta) ou há muitas features com valor comparável (cauda longa)? Isso diz muito sobre a estrutura discriminativa do dataset.
> - Existe alguma feature que aparece no topo aqui, mas estava mal representada nos loadings do PCA (E8)? Isso indicaria que a feature tem alto poder discriminativo, mas pouca variância global.
> - Quantas features têm p-value < 0.05? Isso está alinhado com o que foi observado nos testes de hipótese dos entregáveis anteriores?


### 3.2 Informação Mútua

A **Informação Mútua (MI)** entre uma feature $X$ e o rótulo $Y$ mede o quanto conhecer $X$ reduz a incerteza sobre $Y$:

$$I(X; Y) = H(Y) - H(Y \mid X)$$

onde $H$ é a entropia de Shannon. $I(X; Y) = 0$ significa independência completa; valores maiores indicam maior dependência. Ao contrário do ANOVA, a MI captura relações não-lineares e não exige suposições paramétricas sobre a distribuição dos dados.

Na prática, a implementação do scikit-learn estima a MI por meio de vizinhos mais próximos (estimador de Kraskov), o que a torna computacionalmente mais custosa que o ANOVA, mas capaz de detectar dependências que o F-score deixaria passar.

Features com MI alta e F-score baixo sugerem relações não-lineares com a classe — candidatos interessantes que métodos puramente paramétricos subestimariam.


In [ ]:
# Informação Mútua — estimativa por k-vizinhos (Kraskov)
# n_neighbors=5 é o padrão; discrete_features=False pois as features são contínuas

print('Calculando Informação Mútua (pode levar ~30s)...')
mi_vals = mutual_info_classif(X_treino, y_treino_raw,
                               discrete_features=False,
                               n_neighbors=5,
                               random_state=42)

df_mi = pd.DataFrame({
    'feature' : feature_cols,
    'MI'      : mi_vals,
}).sort_values('MI', ascending=False).reset_index(drop=True)

df_mi['rank_mi'] = df_mi.index + 1

print('Top 20 features por Informação Mútua:')
display(df_mi.head(20).round(5))
print()
print(f'Features com MI > 0.05 : {(df_mi["MI"] > 0.05).sum()}')
print(f'Features com MI > 0.10 : {(df_mi["MI"] > 0.10).sum()}')
print(f'Features com MI = 0    : {(df_mi["MI"] == 0).sum()}')


In [ ]:
# Comparação ANOVA vs. MI — scatter plot de rankings

df_compare_fm = df_anova[['feature', 'F_score', 'rank_anova']].merge(
    df_mi[['feature', 'MI', 'rank_mi']], on='feature'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter rank_anova x rank_mi
axes[0].scatter(df_compare_fm['rank_anova'], df_compare_fm['rank_mi'],
                alpha=0.4, s=15, color='#2980b9')
axes[0].plot([1, len(feature_cols)], [1, len(feature_cols)],
             'r--', lw=1, label='Concordância perfeita')
axes[0].set_xlabel('Rank ANOVA')
axes[0].set_ylabel('Rank Informação Mútua')
axes[0].set_title('Concordância de Rankings: ANOVA vs. MI')
axes[0].legend(fontsize=9)

# Top 20 MI — barplot
top20_mi = df_mi.head(20)
axes[1].barh(top20_mi['feature'][::-1], top20_mi['MI'][::-1],
             color=plt.cm.Blues(np.linspace(0.4, 0.9, 20)))
axes[1].set_xlabel('Informação Mútua (bits)')
axes[1].set_title('Top 20 Features — Informação Mútua')

plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_mi_comparacao.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlação de Spearman entre os dois rankings
from scipy.stats import spearmanr
rho, p_rho = spearmanr(df_compare_fm['rank_anova'], df_compare_fm['rank_mi'])
print(f'Correlação de Spearman entre rankings ANOVA e MI: ρ = {rho:.3f}  (p = {p_rho:.4f})')


**Comentários sobre a subseção 3.2 — Informação Mútua:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - A correlação de Spearman entre os rankings ANOVA e MI ficou alta ou baixa? Uma correlação alta indica que os dois métodos concordam bastante, enquanto uma correlação baixa sugere que features não-lineares têm papel relevante.
> - Há features com MI alta mas F-score relativamente baixo? Essas são as mais interessantes de apontar — indicam que a relação com o rótulo é não-linear e passaria despercebida por um filtro puramente paramétrico.
> - Features de domínio não-linear (entropia, DFA, dimensão fractal) tendem a aparecer melhor ranqueadas na MI do que no ANOVA? Isso seria fisiologicamente coerente.
> - Quantas features têm MI = 0? Isso indica independência estatística com relação à classe — boas candidatas à exclusão já nessa etapa.


### 3.3 ReliefF

O **ReliefF** (Kononenko, 1994) é um algoritmo de seleção baseado na ideia de que uma feature boa deve apresentar valores similares entre instâncias da mesma classe (*near-hits*) e valores diferentes entre instâncias de classes distintas (*near-misses*).

Para cada instância amostrada, o algoritmo identifica os $k$ vizinhos mais próximos de mesma classe e das demais classes, e atualiza o score de cada feature de acordo com a diferença de valores observada. Features cujos valores divergem muito entre vizinhos de classes distintas recebem scores altos; features que divergem dentro da mesma classe são penalizadas.

A versão **ReliefF** (com F) é uma extensão do Relief original para suporte a múltiplas classes — essencial neste contexto, onde temos 5 superclasses diagnósticas.

Diferentemente do ANOVA e da MI, o ReliefF captura **interações locais** entre features, tornando-o especialmente sensível a padrões discriminativos que se manifestam apenas em certas regiões do espaço de features.


In [ ]:
# ReliefF — avaliação baseada em vizinhança

if RELIEFF_OK:
    # Amostragem do treino para custo computacional viável
    # ReliefF tem complexidade O(n_samples^2) — amostramos para não inviabilizar
    N_SAMPLE_RF = min(5000, X_treino.shape[0])
    idx_sample  = np.random.choice(X_treino.shape[0], N_SAMPLE_RF, replace=False)

    X_rf_sample = X_treino[idx_sample]
    y_rf_sample = y_treino_enc[idx_sample]

    print(f'Rodando ReliefF em amostra de {N_SAMPLE_RF} instâncias, k=10 vizinhos...')
    relieff = ReliefF(n_features_to_select=len(feature_cols),
                      n_neighbors=10, n_jobs=-1)
    relieff.fit(X_rf_sample, y_rf_sample)

    df_relieff = pd.DataFrame({
        'feature'       : feature_cols,
        'relieff_score' : relieff.feature_importances_,
    }).sort_values('relieff_score', ascending=False).reset_index(drop=True)

    df_relieff['rank_relieff'] = df_relieff.index + 1

    print('Top 20 features por ReliefF:')
    display(df_relieff.head(20).round(5))

    # Features com score negativo — penalizadas pelo algoritmo
    n_neg = (df_relieff['relieff_score'] < 0).sum()
    print(f'\nFeatures com score negativo (penalizadas): {n_neg} de {len(feature_cols)}')

else:
    print('ReliefF nao disponível. Pulando subseção 3.3.')
    df_relieff = pd.DataFrame({'feature': feature_cols, 'relieff_score': np.nan,
                               'rank_relieff': range(1, len(feature_cols)+1)})


In [ ]:
# Barplot ReliefF — Top 30

if RELIEFF_OK:
    top30_rf = df_relieff.head(30)
    fig, ax  = plt.subplots(figsize=(14, 7))
    cores    = ['#c0392b' if s < 0 else '#27ae60' for s in top30_rf['relieff_score'][::-1]]
    ax.barh(top30_rf['feature'][::-1], top30_rf['relieff_score'][::-1], color=cores)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('ReliefF Score')
    ax.set_title('Top 30 Features — ReliefF (vermelho = score negativo)', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'sel_relieff_top30.png', dpi=150, bbox_inches='tight')
    plt.show()


**Comentários sobre a subseção 3.3 — ReliefF:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quantas features receberam score negativo? Isso indica features que, na vizinhança local, prejudicam a discriminação entre classes — são boas candidatas a exclusão.
> - O ranking ReliefF concorda com ANOVA e MI, ou revela features que os outros métodos subestimaram? Features ranqueadas altas no ReliefF mas baixas nos outros filtros sugerem padrões discriminativos locais e não-lineares.
> - Features não-lineares (entropia, métricas de HRV) aparecem mais proeminentes aqui do que no ANOVA? O ReliefF é estruturalmente mais sensível a esses padrões.
> - O fato de amostrarmos apenas N instâncias para o ReliefF pode introduzir variância no ranking. Vale comentar se a amostragem é representativa dado o tamanho do dataset.


### 3.4 Síntese dos Filter Methods — Ranking Consolidado

Para consolidar os resultados dos três filtros, construímos um **ranking agregado por média de posição** (Borda Count adaptado). Esse método é simples, robusto a outliers de ranking e não exige que as escalas dos scores sejam comparáveis entre os métodos.

O ranking final por filtros serve como referência exploratória e como base de comparação com os métodos wrapper e embedded nas seções seguintes.


In [ ]:
# Consolidação dos rankings de filter methods

df_filter_final = (df_anova[['feature', 'F_score', 'rank_anova']]
                   .merge(df_mi[['feature', 'MI', 'rank_mi']], on='feature')
                   .merge(df_relieff[['feature', 'relieff_score', 'rank_relieff']], on='feature'))

# Rank agregado: média dos três rankings (menor = melhor)
df_filter_final['rank_filter_avg'] = (
    df_filter_final[['rank_anova', 'rank_mi', 'rank_relieff']].mean(axis=1)
)
df_filter_final = df_filter_final.sort_values('rank_filter_avg').reset_index(drop=True)
df_filter_final['rank_filter'] = df_filter_final.index + 1

print('Top 30 features consolidadas pelos Filter Methods (Borda Count):')
display(df_filter_final[['feature', 'F_score', 'MI', 'relieff_score',
                          'rank_anova', 'rank_mi', 'rank_relieff',
                          'rank_filter']].head(30).round(4))


In [ ]:
# Heatmap de concordância entre rankings dos filtros (Top 50 features)

top50_features = df_filter_final.head(50)['feature'].tolist()
df_hm = df_filter_final.set_index('feature').loc[top50_features,
        ['rank_anova', 'rank_mi', 'rank_relieff']]

# Normaliza rankings para [0, 1] — facilita leitura visual
df_hm_norm = df_hm.apply(lambda c: (c - c.min()) / (c.max() - c.min()))

fig, ax = plt.subplots(figsize=(7, 16))
sns.heatmap(df_hm_norm, cmap='RdYlGn_r', ax=ax,
            yticklabels=top50_features,
            xticklabels=['ANOVA', 'MI', 'ReliefF'],
            cbar_kws={'label': 'Rank normalizado (0=melhor, 1=pior)'},
            linewidths=0.2)
ax.set_title('Concordância de Rankings — Filter Methods (Top 50)', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_filter_concordancia_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


---

## 4. Wrapper Methods

### Fundamentação

Os wrapper methods avaliam subconjuntos de features **treinando e validando um estimador** sobre cada subconjunto candidato. Por serem acoplados ao classificador, capturam interações entre features e fornecem uma estimativa de desempenho mais realista do que os filtros — mas com custo computacional significativamente maior.

O estimador utilizado aqui é a **Regressão Logística** com regularização L2 leve (`C=1`), escolhida por ser rápida, estável e interpretável. Classificadores mais complexos (como SVM ou Random Forest) tornaria o processo computacionalmente proibitivo sem hardware dedicado.

A validação é feita por **cross-validation estratificada de 5 folds** sobre o conjunto de treino. O critério de parada nos métodos sequenciais é o número de features a selecionar, fixado em um conjunto de valores a analisar.

> **Nota sobre custo computacional:** SFS e SBE são processos iterativos que avaliam múltiplos subconjuntos de features. Para um dataset com centenas de features, o processo pode ser demorado — o parâmetro `n_features_to_select` deve ser definido com cuidado e, na prática, aplicado sobre um subconjunto pré-filtrado de features candidatas (as top-K dos filter methods).


### 4.1 Sequential Forward Selection (SFS)

O **SFS** começa com um conjunto vazio de features e, a cada iteração, adiciona a feature que mais aumenta o desempenho do estimador (dado o subconjunto já selecionado). O processo continua até atingir o número alvo de features.

O SFS é **guloso** — não reverte decisões anteriores. Isso pode levar a subconjuntos subótimos quando duas features individualmente fracas são coletivamente fortes (a primeira pode não ser selecionada antes que a segunda apareça). Ainda assim, é o método mais comum por seu equilíbrio entre custo e qualidade.

Para tornar o processo viável, aplicamos o SFS sobre as **top-50 features pelos filter methods**, reduzindo o espaço de busca sem comprometer a qualidade do resultado.


In [ ]:
# SFS — aplicado sobre pré-seleção dos filter methods

TOP_K_FILTER = 50   # features pré-filtradas como candidatas ao wrapper
N_SELECT_SFS = 20   # número final de features a selecionar

top_k_features  = df_filter_final.head(TOP_K_FILTER)['feature'].tolist()
idx_top_k       = [feature_cols.index(f) for f in top_k_features]

X_treino_topk   = X_treino[:, idx_top_k]

estimador_sfs = LogisticRegression(C=1, max_iter=500, random_state=42,
                                    solver='saga', multi_class='multinomial')

cv_sfs = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Rodando SFS: {TOP_K_FILTER} candidatas → {N_SELECT_SFS} selecionadas...')
print(f'Estimador  : LogisticRegression(C=1, multinomial)')
print(f'CV         : StratifiedKFold(n_splits=5)')
print()

sfs = SequentialFeatureSelector(
    estimador_sfs,
    n_features_to_select=N_SELECT_SFS,
    direction='forward',
    scoring='accuracy',
    cv=cv_sfs,
    n_jobs=-1
)
sfs.fit(X_treino_topk, y_treino_raw)

# Extrai features selecionadas
mask_sfs      = sfs.get_support()
features_sfs  = [top_k_features[i] for i in range(TOP_K_FILTER) if mask_sfs[i]]

print(f'Features selecionadas pelo SFS ({N_SELECT_SFS}):')
for i, f in enumerate(features_sfs, 1):
    print(f'  {i:2d}. {f}')


**Comentários sobre a subseção 4.1 — SFS:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quais features foram selecionadas? Há concentração em um domínio específico, ou o SFS escolheu features de domínios distintos (temporal + espectral + não-linear)?
> - Alguma feature selecionada pelo SFS estava fora do top-20 dos filter methods? Isso indicaria que a feature tem valor discriminativo em combinação com outras, mas não individualmente.
> - O número de features selecionadas (N_SELECT_SFS = 20) parece adequado para o seu dataset? Experimente valores diferentes e observe se a acurácia de validação satura ou continua crescendo.


### 4.2 Sequential Backward Elimination (SBE)

O **SBE** parte do conjunto completo de features pré-filtradas e, a cada iteração, remove a feature cuja ausência **menos reduz** o desempenho do estimador. O processo continua até atingir o número alvo de features.

O SBE tende a ser mais conservador que o SFS — começa com toda a informação disponível e retira o que é menos necessário, o que pode preservar melhor interações entre features. Por outro lado, é computacionalmente ainda mais custoso em espaços de alta dimensão, pois começa avaliando subconjuntos maiores.

A comparação entre SFS e SBE revela se a seleção de features é sensível à direção da busca — quando ambos concordam, o subconjunto selecionado é mais confiável.


In [ ]:
# SBE — mesma configuração do SFS, direção invertida

N_SELECT_SBE = 20

print(f'Rodando SBE: {TOP_K_FILTER} candidatas → {N_SELECT_SBE} selecionadas...')
print()

sbe = SequentialFeatureSelector(
    estimador_sfs,
    n_features_to_select=N_SELECT_SBE,
    direction='backward',
    scoring='accuracy',
    cv=cv_sfs,
    n_jobs=-1
)
sbe.fit(X_treino_topk, y_treino_raw)

mask_sbe      = sbe.get_support()
features_sbe  = [top_k_features[i] for i in range(TOP_K_FILTER) if mask_sbe[i]]

print(f'Features selecionadas pelo SBE ({N_SELECT_SBE}):')
for i, f in enumerate(features_sbe, 1):
    print(f'  {i:2d}. {f}')


In [ ]:
# Comparação SFS vs. SBE

features_sfs_set = set(features_sfs)
features_sbe_set = set(features_sbe)

intersecao    = features_sfs_set & features_sbe_set
somente_sfs   = features_sfs_set - features_sbe_set
somente_sbe   = features_sbe_set - features_sfs_set

print(f'Interseção SFS ∩ SBE  : {len(intersecao)} features')
print(f'Apenas no SFS         : {len(somente_sfs)} features')
print(f'Apenas no SBE         : {len(somente_sbe)} features')
print()
print('Features presentes em ambos (núcleo robusto):')
for f in sorted(intersecao):
    print(f'  • {f}')
print()
print('Apenas no SFS:')
for f in sorted(somente_sfs): print(f'  • {f}')
print('Apenas no SBE:')
for f in sorted(somente_sbe): print(f'  • {f}')


In [ ]:
# Diagrama de Venn simplificado — sobreposição SFS/SBE

fig, ax = plt.subplots(figsize=(7, 5))

# Representação textual com cores
categorias = {
    'SFS ∩ SBE'  : (len(intersecao), '#2ecc71'),
    'Só SFS'     : (len(somente_sfs), '#3498db'),
    'Só SBE'     : (len(somente_sbe), '#e74c3c'),
}
bars  = list(categorias.keys())
vals  = [v[0] for v in categorias.values()]
cores = [v[1] for v in categorias.values()]

ax.bar(bars, vals, color=cores, edgecolor='white', linewidth=1.5)
for bar_x, val in zip(bars, vals):
    ax.text(bar_x, val + 0.1, str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Número de features')
ax.set_title('Sobreposição SFS vs. SBE')
ax.set_ylim(0, max(vals) * 1.2)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_wrapper_sobreposicao.png', dpi=150, bbox_inches='tight')
plt.show()


**Comentários sobre a subseção 4.2 — SBE e Comparação com SFS:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - A interseção SFS ∩ SBE é grande ou pequena? Uma interseção grande indica robustez — features selecionadas de ambas as direções são mais confiáveis do que as exclusivas de um único método.
> - As features que aparecem somente no SFS (e não no SBE) ou vice-versa tendem a ser de domínios específicos? Isso pode indicar que essas features têm valor discriminativo apenas em certas configurações de subconjunto.
> - O núcleo robusto (interseção) cobre os principais domínios de análise, ou está concentrado em apenas um? Um subconjunto balanceado entre domínios é geralmente mais generalizável.


---

## 5. Embedded Methods

### Fundamentação

Os métodos embedded realizam a seleção de features **durante o treinamento do modelo**, incorporando o critério de seleção à função objetivo do estimador. Isso os torna mais eficientes que os wrappers (não exigem múltiplos retreinos) e mais sensíveis a interações do que os filtros (a seleção é guiada pelo desempenho do modelo).

Dois métodos são aplicados:

- **LASSO (Regularização L1):** impõe uma penalidade proporcional ao valor absoluto dos coeficientes durante o ajuste da regressão logística. Features pouco informativas têm seus coeficientes zerados pelo otimizador — uma forma natural de seleção. O hiperparâmetro de regularização $\lambda$ é escolhido por cross-validation (`LassoCV`).
- **Importância por Random Forest:** durante o treinamento da floresta, cada nó de divisão reduz a impureza (Gini). A importância de cada feature é calculada como a redução média ponderada de impureza ao longo de todas as árvores. Features mais usadas em nós de alta impureza recebem scores maiores.


### 5.1 LASSO (Regularização L1)

No contexto de classificação multiclasse, aplicamos **Regressão Logística com penalidade L1** via `sklearn.linear_model.LogisticRegressionCV`, que faz a busca automática do parâmetro de regularização $C = 1/\lambda$ por cross-validation.

A regularização L1 tende a produzir soluções esparsas — muitos coeficientes vão a zero, realizando seleção implícita. O número de features com coeficiente não-nulo depende da intensidade de regularização: $C$ pequeno (regularização forte) produz mais zeros; $C$ grande (regularização fraca) mantém mais features.

Em problemas multiclasse (5 superclasses), a regressão logística ajusta um vetor de coeficientes por classe ($W \in \mathbb{R}^{C \times F}$). Uma feature é considerada **selecionada** se seu coeficiente for não-nulo em **pelo menos uma** classe.


In [ ]:
# LASSO via Regressão Logística com penalidade L1 e CV automática

from sklearn.linear_model import LogisticRegressionCV

print('Ajustando LASSO (LogisticRegressionCV L1)...')
print('Isso pode levar alguns minutos — solver=saga é o único que suporta L1 multiclasse.')
print()

lasso_cv = LogisticRegressionCV(
    Cs=np.logspace(-3, 1, 15),       # grid de C (= 1/lambda) de 0.001 a 10
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    penalty='l1',
    solver='saga',
    multi_class='multinomial',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
lasso_cv.fit(X_treino, y_treino_raw)

# Coeficientes: shape (n_classes, n_features) para multinomial
coefs     = lasso_cv.coef_            # (5, n_features)
coef_max  = np.abs(coefs).max(axis=0) # max absoluto entre classes para cada feature

mask_lasso   = coef_max > 0
features_lasso_all = [(feature_cols[i], coef_max[i])
                      for i in range(len(feature_cols)) if mask_lasso[i]]
features_lasso_all.sort(key=lambda x: -x[1])

features_lasso = [f for f, _ in features_lasso_all]

print(f'C ótimo (por CV)         : {lasso_cv.C_[0]:.4f}  (lambda = {1/lasso_cv.C_[0]:.4f})')
print(f'Features não zeradas     : {mask_lasso.sum()} de {len(feature_cols)}')
print()
print('Top 20 features por |coeficiente| máximo:')
df_lasso = pd.DataFrame(features_lasso_all, columns=['feature', 'coef_max_abs'])
df_lasso['rank_lasso'] = df_lasso.index + 1
display(df_lasso.head(20).round(5))


In [ ]:
# Coeficientes por classe — heatmap das top-30 features (magnitude)

top30_lasso = df_lasso.head(30)['feature'].tolist()
idx_top30   = [feature_cols.index(f) for f in top30_lasso]
coef_top30  = coefs[:, idx_top30]

labels_abbr = [f.replace('_median','_med').replace('_power','_pwr')
                .replace('wavelet_','wv_').replace('nonlin_','nl_')
                .replace('freq_','fr_').replace('time_','t_')
                .replace('morph_','m_').replace('hrv_','h_')
               for f in top30_lasso]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(coef_top30, cmap='vlag', center=0,
            yticklabels=lasso_cv.classes_,
            xticklabels=labels_abbr,
            cbar_kws={'label': 'Coeficiente L1'},
            linewidths=0.2, ax=ax)
ax.set_title('Coeficientes LASSO por Classe — Top 30 Features', fontsize=12, pad=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_lasso_coeficientes.png', dpi=150, bbox_inches='tight')
plt.show()


**Comentários sobre a subseção 5.1 — LASSO:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - O LASSO zerou muitas features? Um número alto de features zeradas indica que o dataset tem muita redundância, e que o modelo consegue generalizar bem com um subconjunto enxuto.
> - O parâmetro C ótimo ficou próximo de 1 (regularização fraca) ou próximo de 0.001 (regularização muito forte)? Isso diz algo sobre a complexidade do problema.
> - No heatmap de coeficientes por classe, há features com sinais opostos entre classes? Isso indica que a feature é discriminativa em ambos os sentidos — por exemplo, alta para HYP e baixa para NORM.
> - Quais classes têm coeficientes de maior magnitude? Isso pode indicar quais diagnósticos são mais bem caracterizados pelas features disponíveis.


### 5.2 Importância por Random Forest

O **Random Forest** é um ensemble de árvores de decisão treinadas sobre subamostras do dataset e subconjuntos aleatórios de features. A importância de cada feature é calculada como a **redução média ponderada de impureza (Gini)** ao longo de todas as árvores e divisões onde a feature foi utilizada.

Essa métrica tem uma propriedade interessante: captura tanto efeitos principais quanto interações, pois as árvores podem usar múltiplas features em conjunto para definir as divisões. Por outro lado, é conhecida por inflar a importância de features com alta cardinalidade ou alta variância — o que justifica analisá-la em conjunto com os outros métodos.

Utilizamos 300 árvores (`n_estimators=300`), com profundidade máxima controlada para evitar árvores degeneradas, e `class_weight='balanced'` para lidar com o desbalanceamento entre as classes.


In [ ]:
# Random Forest — importância por impureza de Gini

print('Treinando Random Forest para importância de features...')

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_treino, y_treino_raw)

importancias = rf.feature_importances_

df_rf = pd.DataFrame({
    'feature'    : feature_cols,
    'importancia': importancias,
}).sort_values('importancia', ascending=False).reset_index(drop=True)

df_rf['rank_rf'] = df_rf.index + 1

print('Top 20 features por importância (RF):')
display(df_rf.head(20).round(6))
print()
print(f'Acurácia RF no treino (OOB proxy): {rf.score(X_treino, y_treino_raw):.4f}')
print(f'Importância acumulada top-20     : {df_rf.head(20)["importancia"].sum()*100:.1f}%')
print(f'Importância acumulada top-50     : {df_rf.head(50)["importancia"].sum()*100:.1f}%')


In [ ]:
# Barplot Top 30 RF + curva de importância acumulada

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

top30_rf = df_rf.head(30)
ax1.barh(top30_rf['feature'][::-1], top30_rf['importancia'][::-1],
         color=plt.cm.YlOrRd(np.linspace(0.3, 0.95, 30)))
ax1.set_xlabel('Importância (Gini)')
ax1.set_title('Top 30 Features — Random Forest Importance', fontsize=12)

# Importância acumulada
imp_cum = df_rf['importancia'].cumsum()
ax2.plot(range(1, len(imp_cum)+1), imp_cum * 100, '-', color='#8e44ad', lw=2)
ax2.fill_between(range(1, len(imp_cum)+1), imp_cum * 100, alpha=0.1, color='#8e44ad')
for lim in [50, 75, 90, 99]:
    k = int(np.argmax(imp_cum >= lim/100) + 1)
    ax2.axhline(lim, ls='--', lw=1, alpha=0.7)
    ax2.axvline(k, ls='--', lw=1, alpha=0.7)
    ax2.annotate(f'{lim}% → K={k}', xy=(k, lim),
                 xytext=(k + len(feature_cols)*0.04, lim - 3),
                 fontsize=8, arrowprops=dict(arrowstyle='->', lw=1))
ax2.set_xlabel('Número de Features')
ax2.set_ylabel('Importância Acumulada (%)')
ax2.set_title('Importância Acumulada — RF', fontsize=12)
ax2.set_ylim(0, 102)

plt.suptitle('Análise de Importância — Random Forest', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_rf_importancia.png', dpi=150, bbox_inches='tight')
plt.show()


**Comentários sobre a subseção 5.2 — Random Forest:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - A curva de importância acumulada cai rapidamente (muita concentração nas top features) ou é mais gradual? Uma queda rápida indica que poucas features explicam a maior parte da discriminação.
> - Quantas features são necessárias para acumular 90% da importância? Compare com o número de componentes necessários para 95% da variância no PCA (E8) — essa comparação revela diferenças entre compressão e seleção.
> - Features de HRV ou não-lineares aparecem no topo do ranking de RF mas não nos filtros lineares (ANOVA)? Isso seria coerente, pois RF captura interações que ANOVA ignora.
> - O ranking RF concorda com o LASSO? Quando ambos concordam, a evidência de relevância da feature é mais forte.


---

## 6. Validação Estatística por Feature

### Fundamentação

Os métodos de seleção das seções anteriores fornecem rankings e subconjuntos candidatos, mas não garantem formalmente que as diferenças observadas entre classes sejam estatisticamente significativas. Esta seção realiza a **validação estatística individual de cada feature candidata**, aplicando testes de hipótese, correção para múltiplas comparações e cálculo de effect size.

Os três passos desta seção são:

1. **Testes de hipótese por feature:** verificar se as distribuições da feature diferem significativamente entre as classes. Usamos o **Kruskal-Wallis** (extensão não-paramétrica da ANOVA de uma via para $C > 2$ classes), que não exige normalidade e é robusto a assimetrias — características comuns em features de biossinais.

2. **Correção de múltiplas comparações:** ao testar centenas de features simultaneamente, a probabilidade de rejeitar incorretamente $H_0$ aumenta linearmente com o número de testes (problema do *p-value inflation*). Aplicamos duas estratégias de correção: **Bonferroni** (controla o FWER — Family-Wise Error Rate — de forma conservadora) e **FDR de Benjamini-Hochberg** (controla a proporção esperada de falsos positivos — mais liberal e com maior poder estatístico).

3. **Effect size (Eta-quadrado $\eta^2$):** um p-value pequeno confirma que a diferença é real, mas não diz se ela é *relevante*. O effect size quantifica a magnitude do efeito — aqui usamos o **Eta-quadrado** ($\eta^2$), interpretável como a proporção da variância total explicada pela classe.


### 6.1 Testes de Hipótese por Feature — Kruskal-Wallis

O **teste de Kruskal-Wallis** testa a hipótese nula de que as distribuições de uma variável contínua são idênticas entre $C$ grupos. É equivalente não-paramétrico da ANOVA e robusto a violações de normalidade e homocedasticidade — frequentes em features de ECG, especialmente as não-lineares.

$$H = \frac{12}{N(N+1)} \sum_{c=1}^{C} \frac{R_c^2}{n_c} - 3(N+1)$$

onde $R_c$ é a soma dos ranks do grupo $c$ e $n_c$ é o número de amostras nesse grupo. Sob $H_0$, $H$ segue aproximadamente uma distribuição $\chi^2$ com $C-1$ graus de liberdade.

Por eficiência computacional, aplicamos o teste sobre o subconjunto de features presentes em pelo menos um dos métodos de seleção das seções anteriores.


In [ ]:
# Kruskal-Wallis por feature — aplicado sobre candidatas pré-selecionadas

# Reúne todas as features candidatas (união dos métodos anteriores)
candidatas_set = (
    set(df_filter_final.head(80)['feature'].tolist()) |  # top-80 filtros
    set(features_sfs) |
    set(features_sbe) |
    set(df_lasso.head(60)['feature'].tolist()) |
    set(df_rf.head(60)['feature'].tolist())
)
candidatas = sorted(candidatas_set)
print(f'Total de features candidatas para validação estatística: {len(candidatas)}')

# Distribui as amostras por classe (treino)
grupos_por_classe = {
    cls: df_treino.loc[df_treino['primary_class'] == cls, candidatas].values
    for cls in classes_ok
}

resultados_kw = []
for feat in candidatas:
    feat_idx = candidatas.index(feat)
    grupos   = [grupos_por_classe[cls][:, feat_idx] for cls in classes_ok]

    try:
        stat, p = kruskal(*grupos)
    except Exception:
        stat, p = np.nan, np.nan

    resultados_kw.append({'feature': feat, 'H_stat': stat, 'p_kruskal': p})

df_kw = pd.DataFrame(resultados_kw).sort_values('p_kruskal').reset_index(drop=True)

print(f'\nFeatures com p < 0.05 (Kruskal-Wallis) : {(df_kw["p_kruskal"] < 0.05).sum()}')
print(f'Features com p < 0.01                  : {(df_kw["p_kruskal"] < 0.01).sum()}')
print(f'Features com p < 0.001                 : {(df_kw["p_kruskal"] < 0.001).sum()}')
print()
display(df_kw.head(20).round(6))


### 6.2 Correção de Múltiplas Comparações

Ao realizar $M$ testes simultaneamente, a probabilidade de obter pelo menos um falso positivo sob $H_0$ global é:

$$P(\text{ao menos 1 FP}) = 1 - (1 - \alpha)^M$$

Para $M = 200$ e $\alpha = 0.05$: $P(\text{FP}) \approx 99.9\%$. Portanto, corrigir para múltiplas comparações não é uma formalidade — é uma necessidade para que os p-values tenham alguma validade.

Duas estratégias complementares são aplicadas:

- **Bonferroni:** divide o limiar $\alpha$ por $M$ ($\alpha_{\text{corr}} = \alpha / M$). Controla o FWER, mas é muito conservador — features com p-values moderados podem ser descartadas mesmo sendo relevantes.
- **Benjamini-Hochberg (FDR):** controla a proporção esperada de falsos positivos (False Discovery Rate). Mais liberal e com maior poder estatístico que Bonferroni. Em estudos exploratórios como este, é o método preferido.


In [ ]:
# Correção de Bonferroni e FDR (Benjamini-Hochberg)

p_vals_array = df_kw['p_kruskal'].fillna(1.0).values

# Bonferroni
_, p_bonf, _, _  = multipletests(p_vals_array, method='bonferroni', alpha=0.05)

# Benjamini-Hochberg (FDR)
_, p_fdr, _, _   = multipletests(p_vals_array, method='fdr_bh', alpha=0.05)

df_kw['p_bonferroni']  = p_bonf
df_kw['p_fdr_bh']      = p_fdr
df_kw['sig_bonf']      = p_bonf < 0.05
df_kw['sig_fdr']       = p_fdr  < 0.05

print('Resumo após correção para múltiplas comparações:')
print(f'  Significativas por Bonferroni (FWER < 5%) : {df_kw["sig_bonf"].sum()} features')
print(f'  Significativas por FDR-BH (FDR < 5%)      : {df_kw["sig_fdr"].sum()} features')
print()
print('Top 20 features após correção:')
display(df_kw[['feature', 'H_stat', 'p_kruskal',
               'p_bonferroni', 'p_fdr_bh', 'sig_bonf', 'sig_fdr']].head(20).round(6))


In [ ]:
# Visualização — distribuição dos p-values e comparação dos critérios de significância

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma dos p-values brutos
axes[0].hist(df_kw['p_kruskal'].dropna(), bins=40, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(0.05, color='red', ls='--', lw=1.5, label='α = 0.05')
axes[0].set_xlabel('p-value (Kruskal-Wallis bruto)')
axes[0].set_ylabel('Número de features')
axes[0].set_title('Distribuição dos p-values — Kruskal-Wallis')
axes[0].legend(fontsize=9)

# Comparação dos critérios
criterios  = ['p < 0.05 (bruto)', 'Bonferroni < 0.05', 'FDR-BH < 0.05']
n_sig      = [
    (df_kw['p_kruskal'] < 0.05).sum(),
    df_kw['sig_bonf'].sum(),
    df_kw['sig_fdr'].sum()
]
cores_bar = ['#e74c3c', '#e67e22', '#27ae60']
bars = axes[1].bar(criterios, n_sig, color=cores_bar, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, n_sig):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Número de features significativas')
axes[1].set_title('Comparação dos critérios de significância')
axes[1].set_ylim(0, max(n_sig) * 1.2)

plt.suptitle('Validação Estatística — Kruskal-Wallis com Correção de Múltiplas Comparações', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_pvalues_correcao.png', dpi=150, bbox_inches='tight')
plt.show()


**Comentários sobre a subseção 6.2 — Correção de Múltiplas Comparações:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - A correção de Bonferroni reduziu muito o número de features significativas em relação ao FDR? Uma grande diferença indica que muitas features têm p-values moderados (não muito pequenos), o que é típico em datasets com muitas features redundantes.
> - A distribuição dos p-values brutos segue o padrão esperado sob uma mistura de $H_0$ verdadeira e alternativa? (pico perto de 0 para features discriminativas, distribuição uniforme no restante)
> - Com qual critério de correção você vai trabalhar a partir daqui — Bonferroni ou FDR? Justifique: estudos exploratórios tendem a preferir FDR para não perder features potencialmente relevantes.


### 6.3 Effect Size — Eta-quadrado ($\eta^2$)

Um p-value estatisticamente significativo confirma que a diferença existe, mas diz pouco sobre sua **magnitude prática**. Em datasets grandes como o PTB-XL, até diferenças triviais podem atingir significância estatística pelo simples tamanho amostral. O effect size é a medida que responde à pergunta relevante: *a diferença é clinicamente/praticamente relevante?*

O **Eta-quadrado** ($\eta^2$) para o teste de Kruskal-Wallis é calculado como:

$$\eta^2 = \frac{H - (C - 1)}{N - C}$$

onde $H$ é a estatística de Kruskal-Wallis, $C$ é o número de classes e $N$ é o total de observações. Ele representa a proporção da variância total explicada pela variável de classe.

Interpretação convencional (Cohen, 1988):
- $\eta^2 < 0.01$: efeito negligenciável
- $0.01 \leq \eta^2 < 0.06$: efeito pequeno
- $0.06 \leq \eta^2 < 0.14$: efeito médio
- $\eta^2 \geq 0.14$: efeito grande


In [ ]:
# Cálculo do Eta-quadrado para cada feature candidata

C_classes = len(classes_ok)
N_total   = mask_treino.sum()

df_kw['eta_squared'] = (df_kw['H_stat'] - (C_classes - 1)) / (N_total - C_classes)
df_kw['eta_squared'] = df_kw['eta_squared'].clip(lower=0)   # clip em 0 — valores negativos são artefatos numéricos

# Classificação do effect size
def classifica_eta(eta):
    if eta < 0.01:   return 'Negligenciável'
    elif eta < 0.06: return 'Pequeno'
    elif eta < 0.14: return 'Médio'
    else:            return 'Grande'

df_kw['effect_size_cat'] = df_kw['eta_squared'].apply(classifica_eta)

print('Distribuição das features por categoria de effect size:')
display(df_kw['effect_size_cat'].value_counts().rename_axis('Categoria').reset_index(name='N'))

print()
print('Top 20 features por Eta-quadrado (effect size):')
display(
    df_kw.sort_values('eta_squared', ascending=False)
    [['feature', 'H_stat', 'p_fdr_bh', 'eta_squared', 'effect_size_cat']]
    .head(20)
    .round(5)
)


In [ ]:
# Scatter: -log10(p_fdr) vs. eta_squared — equivalente ao volcano plot

df_kw_plot = df_kw.dropna(subset=['p_fdr_bh', 'eta_squared']).copy()
df_kw_plot['log10p'] = -np.log10(df_kw_plot['p_fdr_bh'].clip(lower=1e-300))

# Cores por effect size
cores_es = {
    'Negligenciável': '#bdc3c7',
    'Pequeno'       : '#f39c12',
    'Médio'         : '#e74c3c',
    'Grande'        : '#8e44ad',
}

fig, ax = plt.subplots(figsize=(12, 7))
for cat, cor in cores_es.items():
    mask_c = df_kw_plot['effect_size_cat'] == cat
    ax.scatter(df_kw_plot.loc[mask_c, 'eta_squared'],
               df_kw_plot.loc[mask_c, 'log10p'],
               alpha=0.55, s=25, color=cor, label=cat, rasterized=True)

ax.axhline(-np.log10(0.05), color='gray', ls='--', lw=1, label='FDR = 0.05')
ax.axvline(0.06, color='steelblue', ls=':', lw=1, label='η² = 0.06 (efeito médio)')
ax.axvline(0.14, color='darkred', ls=':', lw=1, label='η² = 0.14 (efeito grande)')
ax.set_xlabel('Effect Size (η²)', fontsize=11)
ax.set_ylabel('-log₁₀(p_FDR)', fontsize=11)
ax.set_title('Volcano Plot — Significância vs. Effect Size por Feature', fontsize=12)
ax.legend(fontsize=9, loc='upper left')

# Anota as 10 features de maior effect size
top10_es = df_kw_plot.nlargest(10, 'eta_squared')
for _, row in top10_es.iterrows():
    ax.annotate(row['feature'].split('_')[0][:10],
                xy=(row['eta_squared'], row['log10p']),
                xytext=(row['eta_squared'] + 0.003, row['log10p'] + 0.5),
                fontsize=7, alpha=0.8)

plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_volcano_effect_size.png', dpi=150, bbox_inches='tight')
plt.show()


**Comentários sobre a subseção 6.3 — Effect Size:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quantas features têm effect size "Médio" ou "Grande"? Essas são as que apresentam diferença prática relevante entre as classes, além de estatisticamente significativas.
> - O volcano plot mostra algum padrão interessante? Por exemplo: features com p-value muito alto mas effect size moderado (pode indicar alta variância intragrupo) ou o contrário (diferença real mas com muitos outliers).
> - As features de effect size grande coincidem com as top-ranked pelos métodos anteriores? Quando há concordância, a evidência de relevância é mais sólida.
> - Há features estatisticamente significativas mas com $\eta^2$ negligenciável? Em datasets grandes como este, isso é comum — são features que você poderia descartar apesar do p-value baixo.


---

## 7. Consolidação e Ranking Final dos Atributos

Com todos os métodos aplicados — três filtros, dois wrappers, dois embedded e validação estatística formal — chegamos à etapa de consolidação. O objetivo é construir um **ranking final unificado** que reflita o consenso entre as diferentes perspectivas de relevância, ponderando tanto a capacidade discriminativa quanto a robustez estatística.

A estratégia adotada é um **Borda Count ponderado**: para cada método, calcula-se o rank da feature naquele critério, e o rank final é a média ponderada dos ranks individuais. Features consistentemente bem ranqueadas em múltiplos métodos são favorecidas sobre aquelas que aparecem no topo de apenas um critério.

Os pesos são atribuídos de acordo com a abrangência do método:
- **Embedded (LASSO, RF):** peso 2 — captura interações e é mais informativo sobre o desempenho do modelo
- **Filter (ANOVA, MI, ReliefF):** peso 1.5 — rápido e abrangente, boa cobertura exploratória
- **Wrapper (SFS, SBE):** peso 2 — avalia subconjuntos, mas com cobertura limitada pelo espaço de busca
- **Effect size (η²):** peso 1.5 — complementa com relevância prática além da estatística


In [ ]:
# Consolidação de todos os rankings em um único DataFrame

# Referência: todas as candidatas que passaram pelo menos por um método
todas_candidatas = sorted(candidatas_set)

df_final = pd.DataFrame({'feature': todas_candidatas})

# --- Filter ---
df_final = df_final.merge(
    df_filter_final[['feature', 'rank_filter']].rename(columns={'rank_filter': 'r_filter'}),
    on='feature', how='left'
)

# --- LASSO ---
df_final = df_final.merge(
    df_lasso[['feature', 'rank_lasso']].rename(columns={'rank_lasso': 'r_lasso'}),
    on='feature', how='left'
)

# --- RF ---
df_final = df_final.merge(
    df_rf[['feature', 'rank_rf']].rename(columns={'rank_rf': 'r_rf'}),
    on='feature', how='left'
)

# --- KW + Effect size ---
df_kw_merge = df_kw.sort_values('eta_squared', ascending=False).reset_index(drop=True)
df_kw_merge['r_eta'] = df_kw_merge.index + 1
df_final = df_final.merge(
    df_kw_merge[['feature', 'r_eta', 'eta_squared', 'sig_fdr', 'p_fdr_bh']],
    on='feature', how='left'
)

# --- Wrapper: features do SFS e SBE ganham rank baseado na interseção ---
# Features na interseção ganham rank 1; só-SFS rank 2; só-SBE rank 3; ausente rank = len+1
def rank_wrapper(feat):
    if feat in intersecao:   return 1
    if feat in somente_sfs:  return 2
    if feat in somente_sbe:  return 2
    return len(todas_candidatas) + 1

df_final['r_wrapper'] = df_final['feature'].apply(rank_wrapper)

# --- Preenche NaN com rank "pior caso" (não apareceu no método) ---
max_rank = len(todas_candidatas) + 1
for col in ['r_filter', 'r_lasso', 'r_rf', 'r_eta']:
    df_final[col] = df_final[col].fillna(max_rank)

# --- Borda Count ponderado ---
pesos = {'r_lasso': 2.0, 'r_rf': 2.0,
         'r_filter': 1.5, 'r_wrapper': 2.0, 'r_eta': 1.5}

soma_pesos = sum(pesos.values())
df_final['rank_final'] = sum(
    df_final[col] * w for col, w in pesos.items()
) / soma_pesos

df_final = df_final.sort_values('rank_final').reset_index(drop=True)
df_final['posicao_final'] = df_final.index + 1

print('Top 30 features pelo ranking consolidado (Borda Count ponderado):')
display(df_final[['posicao_final', 'feature', 'r_filter', 'r_lasso',
                   'r_rf', 'r_wrapper', 'r_eta', 'eta_squared',
                   'sig_fdr', 'rank_final']].head(30).round(3))


In [ ]:
# Heatmap de rankings normalizados — Top 40 features finais

top40_final   = df_final.head(40)
rank_cols     = ['r_filter', 'r_lasso', 'r_rf', 'r_wrapper', 'r_eta']
rank_labels   = ['Filter\n(Borda)', 'LASSO', 'RF\nImportância', 'Wrapper\n(SFS/SBE)', 'Effect\nSize (η²)']

df_hm_final   = top40_final.set_index('feature')[rank_cols].copy()
# Normaliza cada coluna para [0,1] para visualização comparável
df_hm_norm_f  = df_hm_final.apply(lambda c: (c - c.min()) / (c.max() - c.min() + 1e-9))

fig, ax = plt.subplots(figsize=(9, 16))
sns.heatmap(df_hm_norm_f, cmap='RdYlGn_r', ax=ax,
            yticklabels=top40_final['feature'].tolist(),
            xticklabels=rank_labels,
            cbar_kws={'label': 'Rank normalizado (verde = melhor)'},
            linewidths=0.2, annot=False)
ax.set_title('Concordância entre Métodos de Seleção — Top 40 Features', fontsize=12, pad=12)
plt.xticks(fontsize=9)
plt.yticks(fontsize=8, rotation=0)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_ranking_consolidado_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Análise de concordância entre métodos — quantas das top-K estão em consenso?

print('Análise de sobreposição entre os métodos (Top-K features):')
print('-' * 65)

for K in [10, 20, 30, 50]:
    top_k_final   = set(df_final.head(K)['feature'])
    top_k_filter  = set(df_filter_final.head(K)['feature'])
    top_k_lasso   = set(df_lasso.head(K)['feature'])
    top_k_rf      = set(df_rf.head(K)['feature'])
    top_k_eta     = set(df_kw.nlargest(K, 'eta_squared')['feature'])

    # Features presentes em pelo menos 3 dos 4 métodos individuais
    contagens = {}
    for f in top_k_final:
        count = sum([
            f in top_k_filter,
            f in top_k_lasso,
            f in top_k_rf,
            f in top_k_eta,
        ])
        contagens[f] = count

    consenso_3 = sum(v >= 3 for v in contagens.values())
    consenso_4 = sum(v >= 4 for v in contagens.values())

    print(f'  Top-{K:2d}: {consenso_3:2d} features em ≥3 métodos | '
          f'{consenso_4:2d} features em todos os 4 métodos')

print()
print('Features do Top-20 final em CONSENSO (≥ 3 métodos):')
K = 20
top_k_final  = set(df_final.head(K)['feature'])
top_k_filter = set(df_filter_final.head(K)['feature'])
top_k_lasso  = set(df_lasso.head(K)['feature'])
top_k_rf     = set(df_rf.head(K)['feature'])
top_k_eta    = set(df_kw.nlargest(K, 'eta_squared')['feature'])

for f in df_final.head(K)['feature']:
    c = sum([f in top_k_filter, f in top_k_lasso, f in top_k_rf, f in top_k_eta])
    marca = '✓' * c
    print(f'  {f:<55} [{marca}]  ({c}/4 métodos)')


**Comentários sobre a seção 7 — Ranking Consolidado:**

> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quais features formam o núcleo de consenso (presentes em ≥ 3 métodos)? Esse subconjunto é o mais robusto do ponto de vista estatístico e metodológico.
> - O heatmap revela colunas (métodos) com coloração radicalmente diferente das demais? Isso indicaria que aquele método "pensa" de forma muito diferente dos outros — ou captura algo que os demais não capturam.
> - Features de quais domínios (temporal, espectral, não-linear) dominam o top-20 final? Isso tem implicação direta na interpretabilidade clínica do modelo que será treinado no Entregável de RP.
> - Há alguma surpresa no ranking — uma feature que você não esperaria ver tão bem ranqueada, ou ao contrário, uma que parecia importante mas não apareceu no topo? Explorações fisiológicas desse tipo enriquecem muito a análise.


---

## 8. Dataset Final Selecionado e Persistência

### 8.1 Definição do Subconjunto Final de Features

Com base no ranking consolidado e na análise de consenso entre os métodos, definimos o subconjunto final de features que será utilizado pelos classificadores no Entregável de RP.

O critério de corte adotado é: **features presentes entre as top-K do ranking final** e que passaram no teste de significância estatística (FDR-BH < 0.05). O valor de K é definido abaixo e deve equilibrar:

- Poder discriminativo (mais features podem ajudar modelos complexos)
- Regularidade e generalizabilidade (menos features reduzem risco de overfitting)
- Custo computacional nos modelos subsequentes

Adicionalmente, registramos o domínio de cada feature selecionada, possibilitando análise de composição do subconjunto final.


In [ ]:
# Definição do corte final de features

N_FEATURES_FINAL = 30   # ajuste conforme análise de consenso

# Features do ranking final que passaram no critério estatístico
df_sel_final = df_final.head(N_FEATURES_FINAL).merge(
    df_kw[['feature', 'sig_fdr', 'eta_squared', 'p_fdr_bh', 'effect_size_cat']],
    on='feature', how='left'
)

features_finais = df_sel_final['feature'].tolist()

print(f'Subconjunto final: {len(features_finais)} features')
print()

# Classificação por domínio — adapte os prefixos ao seu pipeline
def classifica_dominio(feat):
    if any(feat.startswith(p) for p in ['time_', 't_', 'mav', 'rms', 'zcr', 'hjorth']):
        return 'Temporal'
    if any(feat.startswith(p) for p in ['freq_', 'fr_', 'psd_', 'band_', 'fft']):
        return 'Espectral'
    if any(feat.startswith(p) for p in ['wavelet_', 'wv_', 'stft_', 'hilbert_']):
        return 'Tempo-Frequência'
    if any(feat.startswith(p) for p in ['nonlin_', 'nl_', 'entropy', 'dfa', 'hfd', 'poincare']):
        return 'Não-Linear'
    if any(feat.startswith(p) for p in ['morph_', 'm_', 'qrs_', 'rr_', 'hrv_']):
        return 'Morfológico/HRV'
    if any(feat.startswith(p) for p in ['ratio_', 'rt_', 'delta_', 'norm_', 'nb_']):
        return 'Derivado/Engenhado'
    return 'Outro'

df_sel_final['dominio'] = df_sel_final['feature'].apply(classifica_dominio)

print('Composição por domínio:')
display(df_sel_final['dominio'].value_counts().rename_axis('Domínio').reset_index(name='N'))

print()
print('Dataset final de features selecionadas:')
display(df_sel_final[['posicao_final', 'feature', 'dominio', 'eta_squared',
                       'effect_size_cat', 'sig_fdr', 'rank_final']].round(4))


In [ ]:
# Gráfico de composição por domínio — pizza

composicao = df_sel_final['dominio'].value_counts()
cores_dom  = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    composicao.values,
    labels=composicao.index,
    colors=cores_dom[:len(composicao)],
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.80
)
for t in autotexts: t.set_fontsize(9)
ax.set_title(f'Composição por Domínio — {N_FEATURES_FINAL} Features Selecionadas', fontsize=12)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_composicao_dominio.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.2 Boxplots das Features Selecionadas por Classe

Para uma última verificação visual da separabilidade das features selecionadas, geramos boxplots das distribuições por classe das top-12 features do ranking final. Esse gráfico é especialmente útil para identificar quais classes são mais facilmente separadas pela feature e se há outliers extremos que possam impactar os classificadores.


In [ ]:
# Boxplots das top-12 features selecionadas por classe

top12 = features_finais[:12]
df_box_plot = df_treino[df_treino['primary_class'].isin(classes_ok)].copy()

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.flatten()

palette_cls = dict(zip(classes_ok, sns.color_palette('Set1', len(classes_ok))))

for i, feat in enumerate(top12):
    ax = axes[i]
    sns.boxplot(
        data=df_box_plot, x='primary_class', y=feat,
        order=classes_ok, hue='primary_class',
        palette=palette_cls, legend=False,
        flierprops={'markersize': 2, 'alpha': 0.3},
        ax=ax
    )
    rank_str = f'#{df_sel_final.loc[df_sel_final["feature"]==feat, "posicao_final"].values[0]}'
    eta_str  = f'η²={df_sel_final.loc[df_sel_final["feature"]==feat, "eta_squared"].values[0]:.3f}'
    ax.set_title(f'{feat}\n{rank_str} | {eta_str}', fontsize=8)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=8)
    ax.set_xticklabels(classes_ok, rotation=30, fontsize=8)

plt.suptitle('Distribuição por Classe — Top 12 Features Selecionadas', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'sel_boxplots_top12.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.3 Salvamento dos Artefatos


In [ ]:
# Constrói o dataset final com features selecionadas + metadados

df_selected = df[META_COLS].copy()
df_selected[features_finais] = scaler.transform(df[feature_cols].values)[:, [feature_cols.index(f) for f in features_finais]]

# Salva como parquet — formato padrão do pipeline
df_selected.to_parquet(OUT_DIR / 'features_selected.parquet', index=True)

# Tabela de features selecionadas — útil para documentação e relatório
df_sel_final.to_csv(OUT_DIR / 'features_selected_ranking.csv', index=False)

# Pipeline serializado — necessário para aplicar o mesmo processamento em novos dados
pipeline_sel = {
    'scaler'          : scaler,
    'features_originais': feature_cols,
    'features_selecionadas': features_finais,
    'n_features_final': len(features_finais),
    'metadados_selecao': {
        'metodo_ranking'    : 'Borda Count ponderado (Filter + LASSO + RF + Wrapper + Effect Size)',
        'criterio_corte'    : f'Top-{N_FEATURES_FINAL} com FDR-BH < 0.05',
        'n_candidatas_avaliadas': len(candidatas),
        'n_significativas_fdr': int(df_kw["sig_fdr"].sum()),
    }
}
joblib.dump(pipeline_sel, OUT_DIR / 'feature_selection_pipeline.pkl')

print('Arquivos gerados com sucesso:')
print('-' * 55)
print(f'Dataset selecionado    : features_selected.parquet')
print(f'  - Dimensão           : {df_selected.shape[0]} registros × {len(features_finais)} features')
print()
print(f'Ranking de features    : features_selected_ranking.csv')
print(f'  - Features ranqueadas: {len(df_sel_final)}')
print()
print(f'Pipeline serializado   : feature_selection_pipeline.pkl')
print('-' * 55)


### 8.4 Síntese e Conexão com o Entregável de RP

#### O que foi feito neste entregável

| Etapa                                              | Resultado                                                                      |
| -------------------------------------------------- | ------------------------------------------------------------------------------ |
| ANOVA F-score                                      | Ranking por separação linear de médias entre classes                           |
| Informação Mútua                                   | Ranking por dependência não-linear feature–rótulo                              |
| ReliefF                                            | Ranking por relevância relacional (vizinhança entre instâncias)                |
| Síntese Filter — Borda Count                       | Ranking consolidado dos três filtros                                           |
| Sequential Forward Selection (SFS)                 | Subconjunto de 20 features por adição iterativa                                |
| Sequential Backward Elimination (SBE)              | Subconjunto de 20 features por remoção iterativa                               |
| Comparação SFS ∩ SBE                               | Núcleo robusto de features concordantes nos dois wrappers                      |
| LASSO (LogisticRegressionCV L1)                    | Features com coeficiente não-nulo; ranking por magnitude                       |
| Importância Random Forest                          | Ranking por redução de impureza Gini                                           |
| Kruskal-Wallis por feature                         | Teste de significância estatística não-paramétrico                             |
| Correção: Bonferroni e FDR-BH                      | Controle de múltiplas comparações; identificação de features verdadeiramente significativas |
| Effect size — Eta-quadrado                         | Relevância prática quantificada além do p-value                                |
| Volcano plot                                       | Visualização conjunta de significância e magnitude do efeito                   |
| Ranking final — Borda Count ponderado              | Consolidação de todos os métodos em um único score                             |
| Dataset final (`features_selected.parquet`)        | Subconjunto de {N} features prontas para os classificadores                    |

---

#### Limitações

- Todos os rankings foram calculados sobre o conjunto de treino (folds 1–8). As estimativas podem apresentar algum grau de variância amostral, especialmente para o ReliefF e os wrappers, que operam sobre amostras menores.

- O Borda Count ponderado é um método de agregação heurístico. Pesos diferentes para cada método produziriam subconjuntos diferentes. Essa escolha deve ser validada empiricamente pelos modelos no Entregável de RP.

- O LASSO e o RF capturam relações que os filtros não capturam, mas são sensíveis a hiperparâmetros. A robustez dos resultados pode ser verificada com diferentes valores de $C$ (LASSO) e `n_estimators` / `max_depth` (RF).

- A análise de effect size (η²) assume que diferenças entre medianas são um proxy razoável de relevância clínica. Para problemas onde a forma da distribuição é importante (ex: bimodalidade), essa métrica pode ser insuficiente.

---

#### Próximos passos — Entregável de RP

O `features_selected.parquet` está pronto para entrada nos classificadores. Dois caminhos paralelos podem ser explorados:

1. **Seleção direta (este entregável):** modelos treinados sobre as features originais interpretáveis — cada feature tem significado fisiológico direto.
2. **Compressão PCA (Entregável 8):** modelos treinados sobre os 79 componentes principais — representação mais compacta, sem interpretabilidade individual.

A comparação de desempenho entre esses dois paradigmas é um dos experimentos mais interessantes do Entregável de RP.


In [ ]:
# Tabela resumo final

print('=' * 60)
print('   VERIFICAÇÃO FINAL — ENTREGÁVEL 9')
print('=' * 60)
print(f'  Features originais (E7)       : {len(feature_cols)}')
print(f'  Candidatas avaliadas          : {len(candidatas)}')
print(f'  Significativas (FDR-BH < 5%) : {int(df_kw["sig_fdr"].sum())}')
print(f'  Features selecionadas (final) : {len(features_finais)}')
print(f'  Redução                       : {(1 - len(features_finais)/len(feature_cols))*100:.1f}%')
print()
print('  Composição do subconjunto final:')
for dom, n in df_sel_final['dominio'].value_counts().items():
    print(f'    {dom:<25} : {n} features')
print()
print('  Artefatos gerados:')
print('    - features_selected.parquet')
print('    - features_selected_ranking.csv')
print('    - feature_selection_pipeline.pkl')
print()
print('  Dataset pronto para o Entregável de RP.')
print('=' * 60)
